In [ ]:
from langchain_community.document_loaders import RecursiveUrlLoader
from bs4 import BeautifulSoup
import re

In [ ]:
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()


loader = RecursiveUrlLoader(
    "https://docs.langchain.com/oss/python/langchain/overview", extractor=bs4_extractor)


pages = []
for doc in loader.lazy_load():
    pages.append(doc)

In [ ]:
len(pages)

In [ ]:
# converting list of pages to a single string
document = "\n\n".join([page.page_content for page in pages])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150)
texts = text_splitter.split_text(document)

In [ ]:
len(texts)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2", show_progress=True)
embeddings

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents=pages, embedding=embeddings)

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud",
    temperature=0.2,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        You are a helpful and factual AI assistant.
        Use the following retrieved context to answer the user's question.
        If the answer is not found in the context, reply with:
        "I'm not sure based on the provided information."

        <context>
        {context}
        </context>

        Question: {input}
    """)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

In [15]:
while True:
    user_inp = input("\nAsk a question: ")
    if user_inp.lower() in ["exit", "quit"]:
        break

    response = rag_chain.invoke({"input": user_inp})
    print("\n🧠 AI Answer:")
    print(response["answer"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🧠 AI Answer:
Hello! Based on the provided documentation, I can see this is the LangChain documentation covering topics like Agent Chat UI, Retrieval, and Short-term Memory. 

However, the context doesn't contain specific information to answer a general greeting like "hi." The documentation is focused on technical content about LangChain's features and capabilities.

If you have questions about LangChain, agent development, retrieval-augmented generation (RAG), or any other topics covered in the documentation, I'd be happy to help answer them!
